In [1]:
import pandas as pd
import joblib

model = joblib.load("../models/podium_model.pkl")

train = pd.read_parquet("../data_processed/train.parquet")
val = pd.read_parquet("../data_processed/val.parquet")
test = pd.read_parquet("../data_processed/test.parquet")

feature_cols = [
    'feat_grid', 'feat_driver_form_last3', 'feat_team_form_last3',
    'feat_circuit_history', 'feat_driver_dnf_rate_last5',
    'prev_points', 'prev_standing_position'
]

print("Model and data loaded.")

Model and data loaded.


In [2]:
season_2023 = test[test['year'] == 2023].copy()
print(season_2023.shape)
print(season_2023['round'].nunique(), "rounds")

(440, 42)
22 rounds


In [3]:
X_2023 = season_2023[feature_cols]
season_2023['pred_prob'] = model.predict_proba(X_2023)[:, 1]

In [4]:
season_2023['predicted_rank'] = (
    season_2023.groupby('raceId')['pred_prob']
    .rank(method='first', ascending=False)
)

season_2023[['raceId', 'round', 'driverId', 'pred_prob', 'predicted_rank', 'positionOrder']].sort_values(['round', 'predicted_rank']).head(20)

,raceId,round,driverId,pred_prob,predicted_rank,positionOrder
446,1098,1,830,0.944529,1.0,1
443,1098,1,815,0.917306,2.0,2
451,1098,1,844,0.914027,3.0,19
447,1098,1,832,0.818710,4.0,4
453,1098,1,847,0.725105,5.0,7
440,1098,1,1,0.596209,6.0,5
441,1098,1,4,0.552483,7.0,3
448,1098,1,839,0.322901,8.0,18
452,1098,1,846,0.263905,9.0,17
449,1098,1,840,0.216175,10.0,6


In [5]:
season_2023[season_2023['driverId'] == 844][
    ['round', 'feat_grid', 'prev_standing_position', 'prev_points', 'pred_prob', 'positionOrder']
].head(5)

,round,feat_grid,prev_standing_position,prev_points,pred_prob,positionOrder
451,1,3.0,2.0,308.0,0.914027,19
471,2,12.0,19.0,0.0,0.200273,7
491,3,7.0,8.0,6.0,0.461726,20
511,4,1.0,10.0,6.0,0.793328,3
531,5,7.0,6.0,28.0,0.587182,7


In [6]:
# Official F1 points system for top 10
points_system = {1: 25, 2: 18, 3: 15, 4: 12, 5: 10, 6: 8, 7: 6, 8: 4, 9: 2, 10: 1}

season_2023['simulated_points'] = season_2023['predicted_rank'].map(points_system).fillna(0)

# Sum across the season per driver
predicted_championship = (
    season_2023.groupby('driverId')['simulated_points']
    .sum()
    .sort_values(ascending=False)
)

print(predicted_championship.head(10))

driverId
830    475.0
815    266.0
1      248.0
832    241.0
844    235.0
4      221.0
847    171.0
846    137.0
857     73.0
840     51.0
Name: simulated_points, dtype: float64


In [7]:
drivers = pd.read_csv("../Race data from 1950 to 2026 Race 9/drivers.csv", na_values=['\\N'])

predicted_championship_named = predicted_championship.reset_index().merge(
    drivers[['driverId', 'forename', 'surname']], on='driverId', how='left'
)
predicted_championship_named['driver_name'] = predicted_championship_named['forename'] + ' ' + predicted_championship_named['surname']
print(predicted_championship_named[['driver_name', 'simulated_points']].head(10))

       driver_name  simulated_points
0   Max Verstappen             475.0
1     Sergio Pérez             266.0
2   Lewis Hamilton             248.0
3     Carlos Sainz             241.0
4  Charles Leclerc             235.0
5  Fernando Alonso             221.0
6   George Russell             171.0
7     Lando Norris             137.0
8    Oscar Piastri              73.0
9     Lance Stroll              51.0


In [8]:
driver_standings = pd.read_csv("../Race data from 1950 to 2026 Race 9/driver_standings.csv", na_values=['\\N'])
races = pd.read_csv("../Race data from 1950 to 2026 Race 9/races.csv", na_values=['\\N'])

standings_2023 = driver_standings.merge(races[['raceId', 'year', 'round']], on='raceId')
standings_2023 = standings_2023[standings_2023['year'] == 2023]

final_round = standings_2023['round'].max()
real_final_standings = standings_2023[standings_2023['round'] == final_round].sort_values('points', ascending=False)

real_final_standings_named = real_final_standings.merge(drivers[['driverId', 'forename', 'surname']], on='driverId', how='left')
real_final_standings_named['driver_name'] = real_final_standings_named['forename'] + ' ' + real_final_standings_named['surname']

print(real_final_standings_named[['driver_name', 'points', 'position']].head(10))

       driver_name  points  position
0   Max Verstappen   575.0       1.0
1     Sergio Pérez   285.0       2.0
2   Lewis Hamilton   234.0       3.0
3  Fernando Alonso   206.0       4.0
4  Charles Leclerc   206.0       5.0
5     Lando Norris   205.0       6.0
6     Carlos Sainz   200.0       7.0
7   George Russell   175.0       8.0
8    Oscar Piastri    97.0       9.0
9     Lance Stroll    74.0      10.0


In [9]:
future_2026 = pd.read_parquet("../data_processed/future_2026.parquet")

# Get each driver's most recent race so far this season
latest_snapshot = (
    future_2026.sort_values(['driverId', 'round'])
    .groupby('driverId')
    .tail(1)
    .copy()
)

print(latest_snapshot.shape)
latest_snapshot[['driverId', 'round'] + feature_cols].head(10)

(22, 42)


,driverId,round,feat_grid,feat_driver_form_last3,feat_team_form_last3,feat_circuit_history,feat_driver_dnf_rate_last5,prev_points,prev_standing_position
176,1,9,3.0,2.666667,8.000000,3.000000,0.0,125.0,3.0
177,4,9,21.0,15.666667,17.666667,8.181818,0.0,1.0,18.0
178,807,9,12.0,15.000000,13.000000,8.928571,0.0,0.0,19.0
179,815,9,20.0,16.666667,19.000000,11.692308,0.0,0.0,21.0
180,822,9,18.0,21.333333,19.000000,7.846154,0.0,0.0,20.0
181,830,9,7.0,9.333333,7.333333,6.666667,0.0,73.0,7.0
182,832,9,14.0,16.000000,15.166667,10.333333,0.0,6.0,14.0
183,839,9,17.0,12.666667,14.833333,11.333333,0.0,3.0,16.0
184,840,9,22.0,19.666667,17.666667,10.300000,0.0,0.0,22.0
185,842,9,15.0,7.666667,10.000000,11.777778,0.0,41.0,9.0


In [10]:
latest_snapshot['pred_prob'] = model.predict_proba(latest_snapshot[feature_cols])[:, 1]
latest_snapshot[['driverId', 'pred_prob']].sort_values('pred_prob', ascending=False)

,driverId,pred_prob
194,863,0.940782
176,1,0.885996
188,847,0.868828
186,844,0.856820
187,846,0.694119
190,857,0.649880
196,865,0.618057
181,830,0.456500
197,866,0.285720
191,859,0.274748


In [11]:
top_predictions = latest_snapshot[['driverId', 'pred_prob']].sort_values('pred_prob', ascending=False)
top_predictions_named = top_predictions.merge(drivers[['driverId', 'forename', 'surname']], on='driverId', how='left')
top_predictions_named['driver_name'] = top_predictions_named['forename'] + ' ' + top_predictions_named['surname']
print(top_predictions_named[['driver_name', 'pred_prob']])

              driver_name  pred_prob
0   Andrea Kimi Antonelli   0.940782
1          Lewis Hamilton   0.885996
2          George Russell   0.868828
3         Charles Leclerc   0.856820
4            Lando Norris   0.694119
5           Oscar Piastri   0.649880
6            Isack Hadjar   0.618057
7          Max Verstappen   0.456500
8          Arvid Lindblad   0.285720
9             Liam Lawson   0.274748
10           Pierre Gasly   0.146734
11        Nico Hülkenberg   0.134042
12      Gabriel Bortoleto   0.131867
13           Carlos Sainz   0.119999
14         Oliver Bearman   0.105862
15        Alexander Albon   0.052874
16       Franco Colapinto   0.030620
17           Esteban Ocon   0.029877
18        Valtteri Bottas   0.016847
19           Sergio Pérez   0.011826
20           Lance Stroll   0.011826
21        Fernando Alonso   0.011619


In [12]:
check_ids = top_predictions_named['driverId'].head(5).tolist() + [830]
latest_snapshot[latest_snapshot['driverId'].isin(check_ids)][
    ['forename', 'surname', 'round', 'feat_grid', 'prev_points', 'prev_standing_position', 'pred_prob']
]

,forename,surname,round,feat_grid,prev_points,prev_standing_position,pred_prob
176,Lewis,Hamilton,9,3.0,125.0,3.0,0.885996
181,Max,Verstappen,9,7.0,73.0,7.0,0.456500
186,Charles,Leclerc,9,2.0,79.0,6.0,0.856820
187,Lando,Norris,9,6.0,79.0,5.0,0.694119
188,George,Russell,9,4.0,131.0,2.0,0.868828
194,Andrea Kimi,Antonelli,9,1.0,171.0,1.0,0.940782


In [13]:
#how many rounds has the 2026 season had, and how many are typically in a full season?
print("Races so far in 2026:", future_2026['round'].nunique())

#check a recent complete season for a reasonable total race count
print("Races in 2025:", test[test['year'] == 2025]['round'].nunique() if 2025 in test['year'].values else val[val['year'] == 2025]['round'].nunique())

Races so far in 2026: 9
Races in 2025: 24


In [14]:
latest_snapshot['predicted_rank'] = (
    latest_snapshot['pred_prob']
    .rank(method='first', ascending=False)
)

latest_snapshot[['forename', 'surname', 'pred_prob', 'predicted_rank']].sort_values('predicted_rank')

,forename,surname,pred_prob,predicted_rank
194,Andrea Kimi,Antonelli,0.940782,1.0
176,Lewis,Hamilton,0.885996,2.0
188,George,Russell,0.868828,3.0
186,Charles,Leclerc,0.856820,4.0
187,Lando,Norris,0.694119,5.0
190,Oscar,Piastri,0.649880,6.0
196,Isack,Hadjar,0.618057,7.0
181,Max,Verstappen,0.456500,8.0
197,Arvid,Lindblad,0.285720,9.0
191,Liam,Lawson,0.274748,10.0


In [15]:
points_system = {1: 25, 2: 18, 3: 15, 4: 12, 5: 10, 6: 8, 7: 6, 8: 4, 9: 2, 10: 1}

remaining_races = 24 - 9  # 15

latest_snapshot['points_per_race'] = latest_snapshot['predicted_rank'].map(points_system).fillna(0)
latest_snapshot['projected_additional_points'] = latest_snapshot['points_per_race'] * remaining_races

latest_snapshot['projected_final_points'] = latest_snapshot['prev_points'] + latest_snapshot['projected_additional_points']

final_projection = latest_snapshot[
    ['forename', 'surname', 'prev_points', 'projected_additional_points', 'projected_final_points']
].sort_values('projected_final_points', ascending=False)

print(final_projection)

        forename     surname  prev_points  projected_additional_points  \
194  Andrea Kimi   Antonelli        171.0                        375.0   
176        Lewis    Hamilton        125.0                        270.0   
188       George     Russell        131.0                        225.0   
186      Charles     Leclerc         79.0                        180.0   
187        Lando      Norris         79.0                        150.0   
190        Oscar     Piastri         80.0                        120.0   
181          Max  Verstappen         73.0                         60.0   
196        Isack      Hadjar         42.0                         90.0   
191         Liam      Lawson         30.0                         15.0   
197        Arvid    Lindblad         14.0                         30.0   
185       Pierre       Gasly         41.0                          0.0   
192       Oliver     Bearman         18.0                          0.0   
193       Franco   Colapinto         1